# Solving the Poisson Equation with Firedrake and PETSc

In this notebook, we show how to solve the Poisson equation with Firedrake using various linear solvers available through PETSc.

## Derivation of the Variational Problem

We begin with the strong form of the Poisson equation subject to homogeneous Dirichlet boundary conditions. We seek an unknown function $u$ such that:

$$-\nabla^2 u = f \quad \text{in } \Omega$$
$$u = 0 \quad \text{on } \partial\Omega$$

To obtain the weak (or variational) formulation, we multiply the strong form by a test function $v$ from a suitable function space $V$ (where $v=0$ on $\partial\Omega$) and integrate over the domain $\Omega$:

$$-\int_{\Omega} (\nabla^2 u) v \, dx = \int_{\Omega} f v \, dx$$

Next, we apply integration by parts (Green's first identity) to the left-hand side, yielding:

$$\int_{\Omega} \nabla u \cdot \nabla v \, dx - \int_{\partial\Omega} (\nabla u \cdot \hat{n}) v \, ds = \int_{\Omega} f v \, dx$$

Since our test function $v$ vanishes on the boundary $\partial\Omega$ due to the Dirichlet condition, the boundary integral drops out. We are left with the variational problem: find $u \in V$ such that

$$a(u, v) = L(v) \quad \forall v \in V$$

where the bilinear form $a(u,v)$ and the linear form $L(v)$ are defined as:

$$a(u, v) = \int_{\Omega} \nabla u \cdot \nabla v \, dx$$
$$L(v) = \int_{\Omega} f v \, dx$$

Below, we create a convenience function to solve this problem on a given mesh using continuous Lagrange elements of a given degree. It builds a solver from PETSc options provided as a dictionary.

In [1]:
try:
    import firedrake
except ImportError:
    !wget "https://fem-on-colab.github.io/releases/firedrake-install-release-real.sh" -O "/tmp/firedrake-install.sh" && bash "/tmp/firedrake-install.sh"
    import firedrake

In [2]:
from firedrake import *

def solve_poisson(mesh, degree, params):
    V = FunctionSpace(mesh, "CG", degree)
    uh = Function(V, name="solution")
    
    # boundary conditions
    bcs = [DirichletBC(V, 0, "on_boundary")]
    
    # source term
    f = Constant(1)

    v = TestFunction(V)
    u = TrialFunction(V)
    a = inner(grad(u), grad(v))*dx
    L = inner(f, v)*dx

    problem = LinearVariationalProblem(a, L, uh, bcs)
    solver = LinearVariationalSolver(problem, solver_parameters=params)
    solver.solve()

    its = solver.snes.getLinearSolveIterations()
    return uh, its

### Mesh Hierarchy and Solver Testing

Now we create a function that applies the solver parameters on a sequence of meshes. This sequence is obtained by uniformly refining a $4 \times 4$ mesh of the unit square. We will print the linear solver iteration counts for each level to observe how the solver scales.

In [3]:
def solve_poisson_hierarchy(degree=1, params=None):
    refinements = 4
    base_mesh = UnitSquareMesh(4, 4)
    meshes = MeshHierarchy(base_mesh, refinements)

    print(" Level\t | Iterations")
    print("---------------------")
    for level, mesh in enumerate(meshes):
        uh, its = solve_poisson(mesh, degree, params)
        print(f"   {level}\t | {its}")

### 1. Direct Solver

We start with a direct solver. By definition, a direct solver (like LU factorization using MUMPS) solves the exact matrix system in a single step, so it should always take exactly one iteration regardless of mesh refinement.

In [4]:
lu_params = {
    "ksp_type": "preonly",
    "pc_type": "lu",
    "pc_factor_mat_solver_type": "mumps",
}
solve_poisson_hierarchy(params=lu_params)

 Level	 | Iterations
---------------------
   0	 | 1
   1	 | 1
   2	 | 1
   3	 | 1
   4	 | 1


### 2. Incomplete LU (ILU)

We now try an iterative solver using Conjugate Gradient (CG) preconditioned by an incomplete LU factorization. Observe that as the mesh is refined and the condition number of the system degrades, the number of iterations required to converge increases.

In [5]:
ilu_params = {
    "ksp_type": "cg",
    "pc_type": "ilu",
}
solve_poisson_hierarchy(params=ilu_params)

 Level	 | Iterations
---------------------
   0	 | 4
   1	 | 6
   2	 | 10
   3	 | 17
   4	 | 32


### 3. Algebraic Multigrid (AMG)

Next, we use an Algebraic Multigrid preconditioner (via Hypre). Multigrid methods are optimally scalable for elliptic problems like Poisson, meaning the iteration count should remain roughly constant even as the mesh is refined.

In [6]:
amg_params = {
    "ksp_type": "cg",
    "pc_type": "hypre",
}
solve_poisson_hierarchy(params=amg_params)

 Level	 | Iterations
---------------------
   0	 | 1
   1	 | 3
   2	 | 4
   3	 | 4
   4	 | 4


### 4. Geometric Multigrid (GMG)

Finally, we leverage Firedrake's knowledge of the mesh hierarchy to use Geometric Multigrid. We use Jacobi as the smoother on the fine levels and an exact LU solve on the coarsest level. Like AMG, this should exhibit mesh-independent iteration counts.

In [7]:
gmg_params = {
    "ksp_type": "cg",
    "pc_type": "mg",
    "mg_levels_pc_type": "jacobi",
    "mg_levels_0_ksp_type": "preonly",
    "mg_levels_0_pc_type": "lu",
}
solve_poisson_hierarchy(params=gmg_params)

 Level	 | Iterations
---------------------
   0	 | 1
   1	 | 6
   2	 | 7
   3	 | 6
   4	 | 6
